# TiRex-2 — DIMER forecasting tutorial

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** zero-shot probabilistic time-series forecasting with chronological evaluation

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. The default sample is demonstration evidence, not a production-quality or benchmark claim. No gradient training or fine-tuning occurs.

**Learning objectives:** bootstrap the exact repository revision in a fresh runtime, validate deterministic sample or BYOD input, make a leakage-safe chronological holdout, compare TiRex-2 with a naive last-value baseline, inspect median/model-quantile outputs, and export machine-readable forecasts plus provenance.


## Prerequisites

The reference path uses CPU, avoiding TiRex-2's CUDA compiler/kernel requirements. Optional BYOD upload is gated off by default. BYOD expects a UTF-8 CSV with a unique `timestamp` column and one or more finite numeric target columns in chronological order. Do not upload confidential or restricted data to a hosted notebook environment unless authorized. Uploaded inputs remain in the notebook runtime and are not sent to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

When no repository checkout exists, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The model-facing dependencies are directly pinned. If installation replaces an already imported core package, the cell fails with a restart instruction.

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/tirex-forecasting-pipeline.git'
REPO_NAME = 'tirex-forecasting-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    tracked = {'torch': 'torch', 'numpy': 'numpy', 'pandas': 'pandas'}
    loaded = {module: getattr(sys.modules[module], '__version__', None) for module in tracked.values() if module in sys.modules}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)], check=True)
    stale = []
    for distribution, module in tracked.items():
        if module in loaded and loaded[module] is not None:
            installed = importlib.metadata.version(distribution)
            if loaded[module] != installed:
                stale.append(f'{module}: loaded={loaded[module]}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, numpy as np, pandas as pd, torch
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'device': 'cpu'})

## 2. Create a deterministic sample or load BYOD

The default sample is deterministic synthetic seasonal/trend data. For BYOD, the raw CSV header is inspected before pandas reads the file so duplicate column names cannot be silently renamed. Timestamps must parse, increase strictly, and be regularly spaced for this tutorial's chronological evaluation. The same production target validator used by the pipeline checks target shape and finiteness before baseline or model execution.

In [ ]:
import csv
from tirex_forecasting_pipeline.validation import validate_target

USE_BYOD = False
if USE_BYOD:
    from google.colab import files
    name = next(iter(files.upload()))
    with open(name, newline='', encoding='utf-8-sig') as handle:
        header = next(csv.reader(handle), [])
    if not header or len(header) != len(set(header)):
        raise ValueError('CSV must have non-empty unique column names; duplicate headers are rejected before pandas ingestion')
    if 'timestamp' not in header:
        raise ValueError('BYOD CSV must contain a timestamp column')
    target_columns = [column for column in header if column != 'timestamp']
    if not target_columns:
        raise ValueError('BYOD CSV must contain at least one numeric target column')
    df = pd.read_csv(name)
    timestamps = pd.to_datetime(df['timestamp'], errors='raise')
    if not timestamps.is_monotonic_increasing or timestamps.duplicated().any():
        raise ValueError('timestamps must be unique and strictly increasing')
    if len(timestamps) > 2 and timestamps.diff().dropna().nunique() != 1:
        raise ValueError('timestamps must be regularly spaced for this tutorial path')
    values = df[target_columns].to_numpy(dtype=float).T
else:
    rng = np.random.default_rng(7)
    t = np.arange(256)
    values = (0.02 * t + np.sin(t / 8) + rng.normal(0, 0.05, len(t)))[None, :]
values = validate_target(values)
print({'shape': values.shape, 'sample': 'BYOD' if USE_BYOD else 'deterministic synthetic'})

## 3. Chronological holdout and baseline

The final horizon is held out from the model context; no future target value is passed to the model. Last-value is the naive history-only baseline. This is one tutorial holdout rather than benchmark evidence or an estimate of deployment variability.

In [ ]:
from tirex_forecasting_pipeline import TiRexForecastPipeline, last_value_baseline, mae, rmse, MODEL_ID, MODEL_REVISION
horizon = 32
context = values[:, :-horizon]
truth = values[:, -horizon:]
baseline = last_value_baseline(context, horizon)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'context': context.shape[1], 'horizon': horizon, 'baseline_mae': mae(truth, baseline), 'baseline_rmse': rmse(truth, baseline)})

## 4. Resolve and run TiRex-2

The loader passes the immutable model revision to the upstream TiRex-2 Hugging Face resolver. Output q=0.5 is treated as the median point forecast; all nine model quantiles remain available. The open release is zero-shot: this notebook performs no fine-tuning, streaming adaptation, classification, or regression training.

In [ ]:
pipe = TiRexForecastPipeline.from_pretrained(device='cpu')
result = pipe.forecast(context, horizon=horizon)
pred = result['median']
metrics = {'mae': mae(truth, pred), 'rmse': rmse(truth, pred), 'baseline_mae': mae(truth, baseline), 'baseline_rmse': rmse(truth, baseline)}
print(metrics)

## 5. Export forecasts and provenance

The CSV keeps time-step, variate, median, truth, baseline, and all model quantiles aligned. JSON records repository revision, model identity, runtime, horizon, context, and tutorial metrics.

In [ ]:
import json
os.makedirs('outputs', exist_ok=True)
rows = []
for variate in range(pred.shape[0]):
    for step in range(horizon):
        row = {'variate': variate, 'step': step + 1, 'median': float(pred[variate, step]), 'truth': float(truth[variate, step]), 'last_value_baseline': float(baseline[variate, step])}
        for index, level in enumerate(result['quantile_levels']):
            row[f'q{int(level * 100):02d}'] = float(result['quantiles'][variate, index, step])
        rows.append(row)
pd.DataFrame(rows).to_csv('outputs/tirex_forecast.csv', index=False)
prov = {'repository_revision': REPO_SHA, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'metrics': metrics, 'context_length': context.shape[1], 'horizon': horizon, 'quantile_levels': list(result['quantile_levels']), 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'device': 'cpu'}, 'sample': 'BYOD' if USE_BYOD else 'deterministic synthetic'}
with open('outputs/tirex_provenance.json', 'w', encoding='utf-8') as handle:
    json.dump(prov, handle, indent=2)
print(['outputs/tirex_forecast.csv', 'outputs/tirex_provenance.json'])

## Interpretation and limits

The forecast is zero-shot; no gradient training or fine-tuning occurs. q=0.5 is a model median, and the other quantiles are model quantiles rather than guaranteed confidence intervals. Reported MAE/RMSE come from one chronological tutorial holdout and must be repeated over representative periods before deployment.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Upstream model: https://huggingface.co/NX-AI/TiRex-2
- Upstream code: https://github.com/NX-AI/tirex-2
- Paper: https://arxiv.org/abs/2607.01204
